In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_parquet("../data/cars_clean.parquet")

df["age"] = 2023 - df["year"]
df["miles_per_year"] = df["mileage"] / df["age"].clip(lower=1)

In [12]:
features = ["manufacturer", "model", "age", "mileage", "miles_per_year",
    "drivetrain", "fuel_type", "transmission_type",
    "accidents_or_damage", "one_owner", "personal_use_only",
    "mpg_city", "mpg_hwy", "driver_rating", "driver_reviews_num",
    "seller_rating", "seller_rating_missing"]

cat_cols = ["manufacturer", "model", "drivetrain", "fuel_type", "transmission_type",
            "accidents_or_damage", "one_owner", "personal_use_only"]

df[cat_cols] = df[cat_cols].astype("category")

In [13]:
train, test = train_test_split(df, test_size=0.2, random_state=42)
train_fit, valid = train_test_split(train, test_size=0.1, random_state=42)
print(train_fit.shape, valid.shape, test.shape)

(539147, 22) (59906, 22) (149764, 22)


In [14]:
def score(name, pred):
    err = np.abs(pred - test["price"])
    return {"model": name,
            "MAE": err.mean(),
            "MAPE": (err / test["price"]).mean(),
            "MedianAE": err.median()}

results = []
y_train = np.log(train["price"])
num_cols = [c for c in features if c not in cat_cols]
cat_feats = [c for c in features if c in cat_cols]

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat_feats),
])
ridge = make_pipeline(prep, Ridge(alpha=1.0))
ridge.fit(train[features], y_train)
results.append(score("Ridge", np.exp(ridge.predict(test[features]))))

In [16]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder

rf_prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_feats),
])
rf = make_pipeline(rf_prep, RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                                  n_jobs=-1, random_state=42))
sample = train.sample(100_000, random_state=42)
rf.fit(sample[features], np.log(sample["price"]))
results.append(score("Random Forest (100k sample)", np.exp(rf.predict(test[features]))))

In [17]:
from catboost import CatBoostRegressor

def to_cb(d):
    d = d[features].copy()
    for c in cat_feats:
        d[c] = d[c].astype(str)
    return d

cb = CatBoostRegressor(iterations=3000, learning_rate=0.08, depth=8,
                       loss_function="RMSE", random_seed=42, verbose=500)
cb.fit(to_cb(train_fit), np.log(train_fit["price"]),
       cat_features=cat_feats,
       eval_set=(to_cb(valid), np.log(valid["price"])),
       early_stopping_rounds=100)
results.append(score("CatBoost", np.exp(cb.predict(to_cb(test)))))

0:	learn: 0.5345845	test: 0.5342911	best: 0.5342911 (0)	total: 436ms	remaining: 21m 48s
500:	learn: 0.1369788	test: 0.1359929	best: 0.1359929 (500)	total: 3m 12s	remaining: 16m
1000:	learn: 0.1262979	test: 0.1278250	best: 0.1278250 (1000)	total: 6m 42s	remaining: 13m 23s
1500:	learn: 0.1211643	test: 0.1246083	best: 0.1246083 (1500)	total: 10m 5s	remaining: 10m 4s
2000:	learn: 0.1175760	test: 0.1227675	best: 0.1227675 (2000)	total: 13m 32s	remaining: 6m 45s
2500:	learn: 0.1146591	test: 0.1214176	best: 0.1214143 (2499)	total: 17m 3s	remaining: 3m 24s
2999:	learn: 0.1124553	test: 0.1204546	best: 0.1204546 (2999)	total: 20m 33s	remaining: 0us

bestTest = 0.1204546016
bestIteration = 2999



In [18]:
import lightgbm as lgb

lgbm = lgb.LGBMRegressor(n_estimators=5000, learning_rate=0.05, num_leaves=127,
                         min_child_samples=20, subsample=0.8, subsample_freq=1,
                         colsample_bytree=0.8, random_state=42, verbose=-1)
lgbm.fit(train_fit[features], np.log(train_fit["price"]),
         eval_set=[(valid[features], np.log(valid["price"]))],
         callbacks=[lgb.early_stopping(100), lgb.log_evaluation(500)])
results.append(score("LightGBM", np.exp(lgbm.predict(test[features]))))

c:\Users\Manci\Desktop\GitHub projects\used-car-deal-detector\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[500]	valid_0's l2: 0.0139933
[1000]	valid_0's l2: 0.0136812
[1500]	valid_0's l2: 0.0135435
[2000]	valid_0's l2: 0.0134862
[2500]	valid_0's l2: 0.0134589
Early stopping, best iteration is:
[2435]	valid_0's l2: 0.0134544


In [19]:
pd.DataFrame(results).sort_values("MAPE").round(3)

,model,MAE,MAPE,MedianAE
3,LightGBM,2223.916,0.077,1422.972
2,CatBoost,2484.606,0.082,1526.022
1,Random Forest (100k sample),3105.864,0.104,1854.128
0,Ridge,3331.320,0.111,1940.888


LightGBM had the lowest error on every metric (MAPE 7.8%), ahead of CatBoost (8.3%). Linear regression performed worse than the baseline in dollar terms, because depreciation varies strongly by model, which a linear model can't capture. Random Forest was trained on a 100k sample for speed, so its result is a lower bound. LightGBM is selected as the final model.